# HW2：洪水機率多元線性迴歸

依 CRISP-DM 六階段進行。主資料為 50,000 列、20 個原始特徵，Notebook 呼叫可獨立執行的完整分析程式。請選擇專案的 `.venv` Python kernel，從頭依序執行。

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / "scripts/analyze.py").exists():
    raise RuntimeError("請把 Notebook 工作目錄設為專案根目錄")
sys.path.insert(0, str(ROOT / "scripts"))


## 1. Business Understanding

目標是連續 FloodProbability 分數，不二元化；建立可解釋的教學模型。資料沒有可驗證的地理時間來源，不能直接當作真實防災系統。

## 2. Data Understanding

Kaggle：https://www.kaggle.com/datasets/naiyakhalid/flood-prediction-dataset

實際使用固定 commit 的公開鏡像。來源與 SHA-256 詳見 data/SOURCE.md。

In [2]:
df = pd.read_csv(ROOT / "data/flood.csv")
print("shape:", df.shape, "missing:", df.isna().sum().sum(), "duplicates:", df.duplicated().sum())
print(df.describe().T)


shape: (50000, 21) missing: 0 duplicates: 0
                                   count     mean       std  ...  50%    75%     max
MonsoonIntensity                 50000.0  4.99148  2.236834  ...  5.0  6.000  16.000
TopographyDrainage               50000.0  4.98410  2.246488  ...  5.0  6.000  18.000
RiverManagement                  50000.0  5.01594  2.231310  ...  5.0  6.000  16.000
Deforestation                    50000.0  5.00848  2.222743  ...  5.0  6.000  17.000
Urbanization                     50000.0  4.98906  2.243159  ...  5.0  6.000  17.000
ClimateChange                    50000.0  4.98834  2.226761  ...  5.0  6.000  17.000
DamsQuality                      50000.0  5.01536  2.245000  ...  5.0  6.000  16.000
Siltation                        50000.0  4.98860  2.232642  ...  5.0  6.000  16.000
AgriculturalPractices            50000.0  5.00612  2.234588  ...  5.0  6.000  16.000
Encroachments                    50000.0  5.00638  2.241633  ...  5.0  6.000  18.000
IneffectiveDisasterPr

## 3. Data Preparation

先切分 60/20/20，seed=42。中位數補值、標準化、selector 只根據訓練集。驗證集用於挑 alpha 和模型，測試集只作最終評估。SFS 在訓練子樣本 5,000 列做三折 backward 選擇；fold 內 preprocessing。

## 4. Modeling

以下實際執行五種特徵選擇與三種模型的完整實驗。程式碼可在 scripts/analyze.py 檢閱；不只讀取預先產生結果。

In [3]:
import analyze
analyze.main()
summary = json.loads((ROOT / "results/summary.json").read_text(encoding="utf-8"))


Fitting backward Sequential Feature Selection (training-only n=5000)...
{
  "best": {
    "key": "All20 / LinearRegression",
    "selector": "All20",
    "model": "LinearRegression",
    "features": 20,
    "alpha": null,
    "validation_RMSE": 8.544558348541666e-16,
    "MAE": 6.805611629800978e-16,
    "MSE": 7.305194027504693e-31,
    "RMSE": 8.547042779525965e-16,
    "R2": 1.0,
    "Adjusted_R2": 1.0,
    "MAPE": 1.373611949101299e-13,
    "MAPE_excluded_zeros": 0
  },
  "baseline": {
    "MAE": 0.0398819768,
    "MSE": 0.0025006238011111115,
    "RMSE": 0.05000623762203183,
    "R2": -4.228458193322737e-05,
    "Adjusted_R2": -4.228458193322737e-05,
    "MAPE": 8.11503614791501,
    "MAPE_excluded_zeros": 0
  },
  "diagnostics": [
    {
      "assumption": "\u7dda\u6027\u95dc\u4fc2",
      "test": "Ramsey RESET (powers 2, 3)",
      "statistic": null,
      "p_value": null,
      "note": "\u6b64\u6a21\u578b\u6b98\u5dee\u50c5\u6709\u6d6e\u9ede\u904b\u7b97\u8aa4\u5dee\uff1b\u6539\u

In [4]:
print(pd.read_csv(ROOT / "results/selected_features.csv").to_string(index=False))
print(pd.read_csv(ROOT / "results/validation_search.csv").to_string(index=False))


                          All20               SelectKBest14                           RFE14                           SFS14                     Lasso14                   Permutation14
               MonsoonIntensity          TopographyDrainage                MonsoonIntensity                MonsoonIntensity          TopographyDrainage              TopographyDrainage
             TopographyDrainage             RiverManagement                 RiverManagement              TopographyDrainage             RiverManagement                   Deforestation
                RiverManagement                Urbanization                   ClimateChange                 RiverManagement                Urbanization                   ClimateChange
                  Deforestation                 DamsQuality                     DamsQuality                   ClimateChange                 DamsQuality                     DamsQuality
                   Urbanization                   Siltation                     

## 5. Evaluation

六項指標、五項假設與殘差圖。完整模型殘差近零，四項檢定退化；另呈現 14 特徵模型診斷。未拒絕假設不等於證明成立；無時間／群組資訊時不能確認獨立性。

In [5]:
print(pd.read_csv(ROOT / "results/model_comparison.csv").to_string(index=False))
print("Selected by validation:", summary["best"]["key"])
print("Mean-only baseline:", summary["baseline"])
print("Post-model descriptive identity check:", summary["deterministic_check"])


                             key      selector            model  features     alpha  validation_RMSE          MAE          MSE         RMSE       R2  Adjusted_R2         MAPE  MAPE_excluded_zeros
        All20 / LinearRegression         All20 LinearRegression        20       NaN     8.544558e-16 6.805612e-16 7.305194e-31 8.547043e-16 1.000000     1.000000 1.373612e-13                    0
                   All20 / Ridge         All20            Ridge        20  0.010000     1.680619e-08 1.331412e-08 2.784980e-16 1.668826e-08 1.000000     1.000000 2.709052e-06                    0
                   All20 / Lasso         All20            Lasso        20  0.000001     4.514874e-06 3.576583e-06 2.009868e-11 4.483156e-06 1.000000     1.000000 7.277292e-04                    0
SelectKBest14 / LinearRegression SelectKBest14 LinearRegression        14       NaN     2.746719e-02 2.186726e-02 7.532116e-04 2.744470e-02 0.698778     0.698355 4.416932e+00                    0
           SelectKBe

In [6]:
print(pd.DataFrame(summary["diagnostics"]).to_string(index=False))
print(pd.DataFrame(summary["diagnostics_reduced14"]).to_string(index=False))


assumption                        test  statistic p_value                                      note       conclusion
      線性關係  Ramsey RESET (powers 2, 3)        NaN    None 此模型殘差僅有浮點運算誤差；改參考 SelectKBest14 的非退化殘差診斷。 殘差近零，檢定退化，不作統計解讀
     殘差獨立性 Durbin–Watson (descriptive)        NaN    None 此模型殘差僅有浮點運算誤差；改參考 SelectKBest14 的非退化殘差診斷。 殘差近零，檢定退化，不作統計解讀
      等變異性    Koenker–Breusch–Pagan LM        NaN    None 此模型殘差僅有浮點運算誤差；改參考 SelectKBest14 的非退化殘差診斷。 殘差近零，檢定退化，不作統計解讀
     殘差常態性                 Jarque–Bera        NaN    None 此模型殘差僅有浮點運算誤差；改參考 SelectKBest14 的非退化殘差診斷。 殘差近零，檢定退化，不作統計解讀
  無嚴重多重共線性                 Maximum VIF   1.001002    None     VIF > 5 作為需關注的經驗門檻；共線性影響係數穩定性，不是因果證據。          未見嚴重共線性
assumption                        test  statistic      p_value                                            note     conclusion
      線性關係  Ramsey RESET (powers 2, 3)   0.020490 9.797184e-01                   p < .05 表示有函數形式不足的證據；搭配殘差圖判讀。      未發現足夠違反證據
     殘差獨立性 Durbin–Watson (descriptive)   2.003

### 95% 預測區間

固定訓練資料決定的 SelectKBest14 / OLS，不用驗證目標選此輔助模型，使用 validation 作 split conformal calibration；test 僅評估涵蓋率，未調整寬度。完整模型的近零殘差區間退化，另以 14 特徵區間呈現不確定性。

In [7]:
print(summary["prediction_interval"])
plots = ["actual_vs_predicted.svg", "residuals.svg", "prediction_intervals.svg", "residuals_reduced14.svg", "qq_plot.svg"]
try:
    from IPython.display import display, SVG
    for name in plots:
        display(SVG(filename=str(ROOT / "results" / name)))
except ImportError:
    for name in plots:
        print("SVG plot:", ROOT / "results" / name)


{'model': 'SelectKBest14 / LinearRegression', 'method': 'Split conformal absolute-residual prediction interval', 'confidence': 0.95, 'calibration_count': 10000, 'rank': 9501, 'radius': 0.05461095730292054, 'test_coverage': 0.9497, 'mean_width': 0.10922191460584108, 'note': '固定使用只由 train 決定的 SelectKBest14 / OLS，沒有用 calibration labels 挑此區間模型或 alpha。外層 validation 同時是此固定輔助模型的 calibration；test 未用來調整寬度。名目邊際涵蓋率依賴交換性假設，不保證每個條件或真實地區的涵蓋率。'}
SVG plot: C:\Users\User\VScode\homework\AIoT\HW2_Multiple Linear regression\results\actual_vs_predicted.svg
SVG plot: C:\Users\User\VScode\homework\AIoT\HW2_Multiple Linear regression\results\residuals.svg
SVG plot: C:\Users\User\VScode\homework\AIoT\HW2_Multiple Linear regression\results\prediction_intervals.svg
SVG plot: C:\Users\User\VScode\homework\AIoT\HW2_Multiple Linear regression\results\residuals_reduced14.svg
SVG plot: C:\Users\User\VScode\homework\AIoT\HW2_Multiple Linear regression\results\qq_plot.svg


## 6. Deployment

用同一組訓練參數預測新樣本；完整輸出不裁切到 [0,1]。互動頁使用單一分析區，下拉選單可切換 Flood、50 Startups 或自選 CSV。

In [8]:
from predict import predict_frame
models = json.loads((ROOT / "results/models.json").read_text(encoding="utf-8"))
new = pd.read_csv(ROOT / "data/new_samples.csv")
new["prediction"] = predict_frame(new, models[summary["best"]["key"]])
print(new.to_string(index=False))


 MonsoonIntensity  TopographyDrainage  RiverManagement  Deforestation  Urbanization  ClimateChange  DamsQuality  Siltation  AgriculturalPractices  Encroachments  IneffectiveDisasterPreparedness  DrainageSystems  CoastalVulnerability  Landslides  Watersheds  DeterioratingInfrastructure  PopulationScore  WetlandLoss  InadequatePlanning  PoliticalFactors  prediction
              5.0                 5.0              5.0            5.0           5.0            5.0          5.0        5.0                    5.0            5.0                              5.0              5.0                   5.0         5.0         5.0                          5.0              5.0          5.0                 5.0               5.0       0.500
              8.0                 5.0              5.0            5.0           5.0            5.0          5.0        5.0                    5.0            5.0                              5.0              5.0                   5.0         5.0         5.0            

## 選填項目說明

依老師說明，整個「4. AI 協助要求」為選填，包含 GPT 輔助內容標示、AI 使用與對話紀錄、對話 PDF，以及 NotebookLM 研究與至少 100 字摘要。以上均不列入必做或待完成目標。